# Final Project: Which Economic Environment Is Best for Beginner Investors?

**Subtitle:** How Fed Policy, Inflation, and Recessions Shape Cash and Stock Returns  
**Course:** BUSN 32120 Data Analysis with Python and SQL  
**Project type:** Final project  
**Main tools:** Python, Pandas, Matplotlib, Scikit-learn, SQLite  
**Data sources:** Data sources: FRED macroeconomic datasets, FRED recession indicator dataset, and long-history S&P 500 market dataset from Stooq / Yahoo Finance

## Introduction

The target audience for this analysis is beginner investors and finance students who want to understand how macroeconomic conditions affect simple investment decisions. The project focuses on cash and the S&P 500 because these are two intuitive choices for a beginner investor.

The main question is:

**When are cash-like returns more attractive, and when does the stock market appear more favorable?**

To answer this, the project studies Federal Reserve policy, inflation, unemployment, recession status, and S&P 500 returns. This matters because a high nominal interest rate does not automatically mean cash is attractive. Investors also need to consider inflation. If inflation is higher than the interest rate earned on cash, the real return on cash can still be negative.

The analysis uses Federal Funds Rate, CPI, unemployment, S&P 500, and recession indicator data. Python is used for data cleaning, feature engineering, exploratory data analysis, and modeling. SQL is used separately for validation, joins, aggregations, subqueries, and window functions.

## 1. Import Packages

We use common Python libraries. The workflow intentionally avoids hiding important steps inside complicated functions so that the analysis is easy to follow and explain.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sqlite3

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

plt.rcParams["figure.figsize"] = (11, 6)


## 2. Download Data

To keep the project easy to reproduce, the macro series are downloaded from public FRED CSV links (no private API key required). Each series is downloaded separately and later merged by month.

| Series | Source / ID | Meaning |
|---|---|---|
| Federal Funds Rate | FRED `FEDFUNDS` | Short-term interest rate influenced by Federal Reserve policy |
| Unemployment Rate | FRED `UNRATE` | Percent of the labor force that is unemployed |
| Consumer Price Index | FRED `CPIAUCSL` | Price index used to calculate inflation |
| Recession Indicator | FRED `USREC` | 1 during recession months, 0 otherwise |
| S&P 500 | Stooq `^spx` (Yahoo `^GSPC` / FRED `SP500` as fallbacks) | Stock market index level |

**Note on the S&P 500 source.** FRED's `SP500` series is limited to roughly the last 10 years (an S&P licensing restriction). Using it would shrink the merged dataset to about a decade. To keep the full macro history (back to the 1950s), we download the S&P 500 from Stooq, with Yahoo Finance and then FRED as automatic fallbacks so the notebook always runs even if one source is unavailable.

In [ ]:
def read_fred_csv(series_id, new_column_name):
    """Download one FRED series from the public CSV endpoint (no API key needed)."""
    url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}"
    df = pd.read_csv(url)
    df.columns = ["date", new_column_name]
    df["date"] = pd.to_datetime(df["date"])
    df[new_column_name] = pd.to_numeric(df[new_column_name], errors="coerce")
    return df


def read_sp500_long():
    """Download a LONG history of the S&P 500 index level.

    FRED's SP500 series only covers the most recent ~10 years (an S&P licensing
    limit), which would shrink the merged sample to about a decade. To keep the
    full macro history, we pull the S&P 500 from a longer source: Stooq first,
    then Yahoo Finance, and finally FRED as a fallback so the notebook always runs.
    Returns a tidy DataFrame with columns ["date", "sp500"].
    """
    # 1) Stooq: monthly S&P 500 as a public CSV (no key, decades of history)
    try:
        stooq = pd.read_csv("https://stooq.com/q/d/l/?s=^spx&i=m")
        df = stooq[["Date", "Close"]].rename(columns={"Date": "date", "Close": "sp500"})
        df["date"] = pd.to_datetime(df["date"])
        df["sp500"] = pd.to_numeric(df["sp500"], errors="coerce")
        df = df.dropna()
        if len(df) > 200:
            print(f"S&P 500 source: Stooq ({len(df)} monthly rows, "
                  f"{df['date'].min().date()} to {df['date'].max().date()})")
            return df
    except Exception as exc:
        print("Stooq download failed, trying Yahoo Finance:", exc)

    # 2) Yahoo Finance via yfinance
    try:
        import yfinance as yf
        raw = yf.download("^GSPC", start="1950-01-01", progress=False, auto_adjust=False)
        close = raw["Close"]
        if hasattr(close, "columns"):          # newer yfinance returns a DataFrame
            close = close.iloc[:, 0]
        monthly = close.resample("MS").last().dropna()
        df = monthly.reset_index()
        df.columns = ["date", "sp500"]
        print(f"S&P 500 source: Yahoo Finance ({len(df)} monthly rows)")
        return df
    except Exception as exc:
        print("Yahoo Finance download failed, falling back to FRED (last ~10 years only):", exc)

    # 3) Fallback: FRED SP500 (only the most recent ~10 years)
    return read_fred_csv("SP500", "sp500")


fed_funds = read_fred_csv("FEDFUNDS", "fed_funds_rate")
unemployment = read_fred_csv("UNRATE", "unemployment_rate")
cpi = read_fred_csv("CPIAUCSL", "cpi_index")
sp500_raw = read_sp500_long()
recession = read_fred_csv("USREC", "recession")

print("Fed Funds rows:", len(fed_funds))
print("Unemployment rows:", len(unemployment))
print("CPI rows:", len(cpi))
print("S&P 500 rows:", len(sp500_raw))
print("Recession rows:", len(recession))

## 3. Prepare Monthly Data and Merge

The macro series are reported monthly, while the S&P 500 source is higher-frequency. To merge everything cleanly, every series is converted to one observation per month. For the monthly macro variables this simply keeps the monthly value; for the S&P 500 we keep the last value in each month so that monthly returns can be calculated.

**Merge design.** The four macro series (Federal Funds Rate, unemployment, CPI, and the recession flag) all reach back to at least the 1950s, so they are inner-joined into a single macro "anchor" table that spans the full history. The S&P 500 is then attached with a **left join** rather than an inner join. This matters: an inner join would shrink the entire dataset to the S&P 500 window — only about a decade if the download had to fall back to FRED's licensing-limited `SP500` series — which would leave almost no recessions and only one or two decades to analyze. With a left join, the S&P 500 is simply missing (`NaN`) for the oldest months, while the macro history stays intact, so the decade and recession analysis always covers the full sample.

In [ ]:
def to_monthly(df, value_col, method="last"):
    """Convert a FRED dataset to one observation per month."""
    temp = df.dropna(subset=[value_col]).copy()
    temp["month"] = temp["date"].dt.to_period("M").dt.to_timestamp()

    if method == "last":
        monthly = (
            temp.sort_values("date")
            .groupby("month", as_index=False)[value_col]
            .last()
        )
    elif method == "mean":
        monthly = temp.groupby("month", as_index=False)[value_col].mean()
    else:
        raise ValueError("method must be 'last' or 'mean'")

    monthly = monthly.rename(columns={"month": "date"})
    return monthly[["date", value_col]]

fed_funds_m = to_monthly(fed_funds, "fed_funds_rate", method="last")
unemployment_m = to_monthly(unemployment, "unemployment_rate", method="last")
cpi_m = to_monthly(cpi, "cpi_index", method="last")
sp500_m = to_monthly(sp500_raw, "sp500", method="last")
recession_m = to_monthly(recession, "recession", method="last")

# Build the macro "anchor" table from the four long-history series with INNER joins.
# Federal Funds, unemployment, CPI, and the recession flag all go back to at least
# the 1950s, so their overlap spans roughly 1954 to today (~860 months).
macro = fed_funds_m.merge(unemployment_m, on="date", how="inner")
macro = macro.merge(cpi_m, on="date", how="inner")
macro = macro.merge(recession_m, on="date", how="inner")

# LEFT JOIN the S&P 500 onto the macro anchor. This is the key design choice:
# the S&P 500 simply becomes NaN for the oldest months when the chosen source is
# short, but the macro history is NEVER truncated. An inner join here would shrink
# the whole sample to the S&P 500 window (only ~10 years if the FRED fallback is
# used), which would gut the decade- and recession-level analysis. The left join
# guarantees the full macro sample survives no matter which S&P 500 source won.
econ = macro.merge(sp500_m, on="date", how="left")
econ = econ.sort_values("date").reset_index(drop=True)

sp500_months = econ["sp500"].notna().sum()
print("Merged dataset date range:", econ["date"].min().date(), "to", econ["date"].max().date())
print("Rows in merged dataset (full macro history):", len(econ))
print(f"Months with S&P 500 data: {sp500_months} "
      f"({sp500_months / len(econ) * 100:.0f}% of the sample)")
econ.head()

## 4. Data Quality Checks

Before analysis, we check missing values, duplicate dates, data types, basic summary statistics, and unusual values. In macroeconomic data, extreme values should not be automatically removed because they may represent real historical events, such as recessions, inflation shocks, or policy tightening cycles.

One block of missing values is **expected by design**: because the S&P 500 was left-joined onto the longer macro history, the `sp500` column (and any feature derived from it) is missing for the oldest months that predate the available S&P 500 data. This is intentional — those rows are kept so the macro analysis covers the full sample, and the stock-return models simply restrict themselves to the months where S&P 500 data exists.

In [ ]:
# Check data types and non-null counts.
econ.info()


In [ ]:
# Table 1: Missing values table.
missing_table = pd.DataFrame({
    "missing_count": econ.isna().sum(),
    "missing_percent": econ.isna().mean() * 100
}).round(2)
missing_table


In [ ]:
# Check duplicate dates.
duplicate_dates = econ["date"].duplicated().sum()
duplicate_dates


In [ ]:
# Table 2: Summary statistics.
summary_stats = econ.describe().round(2)
summary_stats


In [ ]:
# Check potential oddities by reviewing the highest Federal Funds Rate months.
econ[["date", "fed_funds_rate", "unemployment_rate", "cpi_index", "sp500", "recession"]].sort_values(
    "fed_funds_rate", ascending=False
).head(10)


## 5. Feature Engineering

We create new columns that make the project more investor-focused:

1. **Year-over-year inflation:** annual inflation rate based on CPI.
2. **Real interest rate:** Federal Funds Rate minus inflation.
3. **Monthly real cash return proxy:** annual real rate divided by 12.
4. **Monthly rate change:** change in the Federal Funds Rate from the previous month.
5. **S&P 500 monthly return:** percentage change in the S&P 500.
6. **S&P 500 12-month return:** one-year stock market return.
7. **Stock-minus-cash gap:** S&P 500 monthly return minus monthly real cash return proxy.
8. **Negative real rate dummy:** equals 1 when the real rate is negative.
9. **Rate hiking dummy:** equals 1 when the Federal Funds Rate increased from the previous month.
10. **High inflation dummy:** equals 1 when inflation is above 3%.
11. **Readable categories:** created with `map()` and `apply()` for recession labels, real-rate environments, inflation environments, and investment environments.


In [ ]:
econ["inflation_yoy"] = ((econ["cpi_index"] / econ["cpi_index"].shift(12)) - 1) * 100
econ["real_interest_rate"] = econ["fed_funds_rate"] - econ["inflation_yoy"]
econ["monthly_real_cash_return_proxy"] = econ["real_interest_rate"] / 12

econ["rate_change"] = econ["fed_funds_rate"].diff()
econ["sp500_return_mom"] = econ["sp500"].pct_change() * 100
econ["sp500_return_yoy"] = econ["sp500"].pct_change(12) * 100
econ["stock_minus_cash_gap"] = econ["sp500_return_mom"] - econ["monthly_real_cash_return_proxy"]

econ["negative_real_rate"] = np.where(econ["real_interest_rate"] < 0, 1, 0)
econ["rate_hiking_month"] = np.where(econ["rate_change"] > 0, 1, 0)
econ["high_inflation"] = np.where(econ["inflation_yoy"] > 3, 1, 0)
econ["decade"] = (econ["date"].dt.year // 10) * 10

# Use map to create readable labels.
econ["recession_label"] = econ["recession"].map({0: "Expansion", 1: "Recession"})
econ["high_inflation_label"] = econ["high_inflation"].map({0: "Low or Moderate Inflation", 1: "High Inflation"})

# Use apply to create readable real-rate and investor environments.
def classify_real_rate(x):
    if pd.isna(x):
        return np.nan
    elif x < 0:
        return "Negative real rate"
    else:
        return "Positive real rate"


def classify_investment_environment(row):
    if pd.isna(row["real_interest_rate"]) or pd.isna(row["inflation_yoy"]):
        return np.nan
    if row["real_interest_rate"] >= 0 and row["inflation_yoy"] <= 3:
        return "Cash-friendly"
    elif row["real_interest_rate"] < 0 and row["inflation_yoy"] > 3:
        return "Inflation pressure"
    elif row["recession"] == 1:
        return "Recession stress"
    else:
        return "Mixed environment"


econ["real_rate_category"] = econ["real_interest_rate"].apply(classify_real_rate)
econ["investment_environment"] = econ.apply(classify_investment_environment, axis=1)

econ.head(15)


## 6. Additional Data Quality Checks After Feature Engineering

After creating new variables, we check missing values again. Some missing values are expected because year-over-year inflation and 12-month stock returns require 12 months of prior data.


In [ ]:
feature_missing_table = pd.DataFrame({
    "missing_count": econ.isna().sum(),
    "missing_percent": econ.isna().mean() * 100
}).round(2)
feature_missing_table


In [ ]:
# Table 3: Unique values for categorical variables.
unique_values_table = pd.DataFrame({
    "unique_values": econ[[
        "decade",
        "recession",
        "recession_label",
        "real_rate_category",
        "high_inflation_label",
        "investment_environment"
    ]].nunique()
})
unique_values_table


In [ ]:
# Check the engineered variables for reasonable ranges.
engineered_summary = econ[[
    "inflation_yoy",
    "real_interest_rate",
    "monthly_real_cash_return_proxy",
    "sp500_return_mom",
    "sp500_return_yoy",
    "stock_minus_cash_gap"
]].describe().round(2)
engineered_summary


## 7. Exploratory Data Analysis Tables

These tables summarize the data in ways that are directly relevant to the target audience. Instead of only describing macro variables, the tables compare investor-relevant environments: decade, recession status, real interest rate status, inflation status, and broader investment environment.


In [ ]:
# Aggregate Table 1: Macro and market conditions by decade.
decade_summary = (
    econ.groupby("decade")
    .agg(
        months=("date", "count"),
        avg_fed_funds=("fed_funds_rate", "mean"),
        avg_inflation=("inflation_yoy", "mean"),
        avg_unemployment=("unemployment_rate", "mean"),
        avg_real_rate=("real_interest_rate", "mean"),
        avg_sp500_monthly_return=("sp500_return_mom", "mean"),
        avg_stock_minus_cash_gap=("stock_minus_cash_gap", "mean"),
        recession_months=("recession", "sum")
    )
    .round(2)
)
decade_summary


### Interpretation of Aggregate Table 1

This decade summary shows that the investor environment changes a lot across time. The 1970s had almost the same average Fed Funds Rate and inflation rate, so the average real rate was close to zero. The 1980s were very different: nominal rates were extremely high, but inflation was lower than the policy rate, so the average real rate was strongly positive. The 2010s and 2020s had negative average real rates in this sample, meaning cash-like returns were weak after adjusting for inflation.

For beginner investors, this table explains why we should not judge cash only by the headline interest rate. The same nominal rate can mean very different things depending on inflation and the decade-specific economic regime.


In [ ]:
# Aggregate Table 2: Recession vs expansion summary.
recession_summary = (
    econ.groupby("recession_label")
    .agg(
        months=("date", "count"),
        avg_fed_funds=("fed_funds_rate", "mean"),
        avg_inflation=("inflation_yoy", "mean"),
        avg_unemployment=("unemployment_rate", "mean"),
        avg_real_rate=("real_interest_rate", "mean"),
        avg_sp500_monthly_return=("sp500_return_mom", "mean"),
        avg_stock_minus_cash_gap=("stock_minus_cash_gap", "mean")
    )
    .round(2)
)
recession_summary


### Interpretation of Aggregate Table 2

This table compares expansion and recession months. Recession months have higher average unemployment and higher average inflation in the full sample. The average S&P 500 monthly return is also much lower during recession months than during expansion months, although the S&P 500 portion is based only on months with available stock data.

For the target audience, the practical lesson is that recession status gives useful context. A beginner investor should understand that recessions are not just a labor-market event; they are also periods when stock returns and cash-vs-stock tradeoffs can look different.


In [ ]:
# Aggregate Table 3: Positive vs negative real interest rate summary.
real_rate_summary = (
    econ.groupby("real_rate_category")
    .agg(
        months=("date", "count"),
        avg_fed_funds=("fed_funds_rate", "mean"),
        avg_inflation=("inflation_yoy", "mean"),
        avg_unemployment=("unemployment_rate", "mean"),
        avg_sp500_monthly_return=("sp500_return_mom", "mean"),
        avg_monthly_real_cash_return=("monthly_real_cash_return_proxy", "mean"),
        avg_stock_minus_cash_gap=("stock_minus_cash_gap", "mean"),
        recession_months=("recession", "sum")
    )
    .round(2)
)
real_rate_summary


### Interpretation of Aggregate Table 3

This table directly connects the project to the cash decision. When the real rate is negative, the monthly real cash return proxy is negative on average. When the real rate is positive, the cash proxy is positive on average. This supports the main project idea that cash is attractive only when the interest rate is high enough to beat inflation.

For beginner investors, this is the most intuitive result: a bank account can show a positive nominal yield, but the investor can still lose purchasing power if inflation is higher.


In [ ]:
# Aggregate Table 4: High inflation vs low/moderate inflation summary.
inflation_environment_summary = (
    econ.groupby("high_inflation_label")
    .agg(
        months=("date", "count"),
        avg_fed_funds=("fed_funds_rate", "mean"),
        avg_real_rate=("real_interest_rate", "mean"),
        avg_sp500_monthly_return=("sp500_return_mom", "mean"),
        avg_stock_minus_cash_gap=("stock_minus_cash_gap", "mean"),
        recession_months=("recession", "sum")
    )
    .round(2)
)
inflation_environment_summary


### Interpretation of Aggregate Table 4

High-inflation months have a much higher average Fed Funds Rate than low or moderate inflation months. This makes economic sense because the Fed often raises rates when inflation pressure is high. The stock-minus-cash gap remains positive in both groups in the available S&P 500 sample, but it should be interpreted carefully because the market-return sample is shorter than the macro sample.

For the target audience, this table shows that inflation changes the meaning of both cash and stocks. High inflation does not automatically make cash attractive, because the investor still needs to compare the nominal rate against inflation.


In [ ]:
# Aggregate Table 5: Investor-oriented environment summary.
investment_environment_summary = (
    econ.groupby("investment_environment")
    .agg(
        months=("date", "count"),
        avg_fed_funds=("fed_funds_rate", "mean"),
        avg_inflation=("inflation_yoy", "mean"),
        avg_real_rate=("real_interest_rate", "mean"),
        avg_sp500_monthly_return=("sp500_return_mom", "mean"),
        avg_monthly_real_cash_return=("monthly_real_cash_return_proxy", "mean"),
        avg_stock_minus_cash_gap=("stock_minus_cash_gap", "mean")
    )
    .sort_values("avg_stock_minus_cash_gap", ascending=False)
    .round(2)
)
investment_environment_summary


### Interpretation of Aggregate Table 5

This investor-oriented table translates macro data into easier categories. The “cash-friendly” environment has positive real rates and low or moderate inflation, which is the most favorable setup for cash. The “inflation pressure” environment has negative average real rates, so cash loses purchasing power on average. “Recession stress” has weak S&P 500 monthly returns in the available market sample, which makes sense because recession months are usually more difficult for risky assets.

This table is useful for beginner investors because it converts abstract macro indicators into plain investment environments. It helps answer the project question in a more practical way.


## 8. Exploratory Data Analysis Charts

The charts are designed to answer the main project question visually. They show how rates, inflation, unemployment, real rates, S&P 500 returns, recession status, and investor-relevant environments move together.

All charts use Matplotlib directly instead of Pandas built-in plotting.


In [ ]:
# Chart 1: Fed Funds Rate, inflation, and unemployment over time.
plt.figure(figsize=(12, 6))
plt.plot(econ["date"], econ["fed_funds_rate"], label="Fed Funds Rate")
plt.plot(econ["date"], econ["inflation_yoy"], label="Inflation YoY")
plt.plot(econ["date"], econ["unemployment_rate"], label="Unemployment Rate")
plt.title("Fed Funds Rate, Inflation, and Unemployment Over Time")
plt.xlabel("Date")
plt.ylabel("Percent")
plt.legend()
plt.grid(True)
plt.show()


### Chart 1 Interpretation

This chart shows that the Fed Funds Rate, inflation, and unemployment move through very different cycles over time. Inflation and interest rates were especially high around the late 1970s and early 1980s, while the post-2008 period had very low interest rates. Unemployment spikes during stress periods, including recessions and the COVID period.

For beginner investors, the chart gives the big-picture context. Cash returns are not decided by interest rates alone. They depend on the broader macro environment, especially whether inflation is also high.


In [ ]:
# Chart 2: Real interest rate over time.
plt.figure(figsize=(12, 6))
plt.plot(econ["date"], econ["real_interest_rate"], label="Real Interest Rate")
plt.axhline(0, linestyle="--", label="Zero Real Rate Line")
plt.title("Real Interest Rate Over Time")
plt.xlabel("Date")
plt.ylabel("Percent")
plt.legend()
plt.grid(True)
plt.show()


### Chart 2 Interpretation

The real interest rate moves above and below zero over time. A positive real rate means cash-like returns are more likely to protect purchasing power. A negative real rate means inflation is higher than the nominal rate, so cash quietly loses real value. In this notebook output, about one-third of valid months have a negative real rate.

This chart is central to the target audience. It shows why a “high” savings rate can still be bad if inflation is even higher. Beginner investors should evaluate cash in real terms, not just nominal terms.


In [ ]:
# Chart 3: S&P 500 monthly return over time.
plt.figure(figsize=(12, 6))
plt.plot(econ["date"], econ["sp500_return_mom"], label="S&P 500 Monthly Return")
plt.axhline(0, linestyle="--", label="Zero Return Line")
plt.title("S&P 500 Monthly Returns Over Time")
plt.xlabel("Date")
plt.ylabel("Monthly Return (%)")
plt.legend()
plt.grid(True)
plt.show()


### Chart 3 Interpretation

This chart shows monthly S&P 500 returns in the available stock-market sample. The main pattern is volatility: monthly stock returns move up and down much more sharply than the cash proxy. This supports the idea that stocks may offer higher average returns, but they are much less stable month to month.

For beginner investors, this chart helps separate two ideas. Cash is about stability and purchasing power, while stocks involve more short-term risk. The S&P 500 analysis should also be interpreted with the sample limitation in mind, because this v4 run only has S&P 500 data for the months available after the fallback data source.


In [ ]:
# Chart 4: Average macro indicators by decade using Matplotlib.
chart4_data = decade_summary[["avg_fed_funds", "avg_inflation", "avg_unemployment"]].dropna()
x = np.arange(len(chart4_data.index))
width = 0.25

plt.figure(figsize=(12, 6))
plt.bar(x - width, chart4_data["avg_fed_funds"], width, label="Fed Funds Rate")
plt.bar(x, chart4_data["avg_inflation"], width, label="Inflation")
plt.bar(x + width, chart4_data["avg_unemployment"], width, label="Unemployment")
plt.title("Average Macro Indicators by Decade")
plt.xlabel("Decade")
plt.ylabel("Percent")
plt.xticks(x, chart4_data.index.astype(str))
plt.legend()
plt.grid(True, axis="y")
plt.show()


### Chart 4 Interpretation

This decade bar chart shows that macro regimes are very different across decades. The 1970s and 1980s stand out because interest rates and inflation were much higher than in later periods. The 2010s had very low interest rates and relatively low inflation, while the 2020s show inflation pressure returning.

For the target audience, this chart reinforces that investment decisions are regime-dependent. A cash strategy that makes sense in the early 1980s may not make sense in a low-rate or negative-real-rate period.


In [ ]:
# Chart 5: S&P 500 monthly returns during expansions and recessions using Matplotlib.
plot_data = econ.dropna(subset=["sp500_return_mom", "recession_label"])
labels = ["Expansion", "Recession"]
box_values = [
    plot_data.loc[plot_data["recession_label"] == label, "sp500_return_mom"]
    for label in labels
]

plt.figure(figsize=(9, 6))
plt.boxplot(box_values)
plt.xticks([1, 2], labels)   # set tick labels separately so it works on any Matplotlib version
plt.title("S&P 500 Monthly Returns: Expansion vs Recession")
plt.xlabel("Economic State")
plt.ylabel("Monthly S&P 500 Return (%)")
plt.grid(True, axis="y")
plt.show()

### Chart 5 Interpretation

This chart compares S&P 500 monthly returns in expansion and recession months. The visual takeaway is that stock returns tend to look weaker during recession periods than during expansions. However, this result should be interpreted carefully because the S&P 500 sample in this v4 run is much shorter than the full macro sample, and recession observations are limited.

For beginner investors, the chart shows why economic state matters. Recessions do not give a perfect trading rule, but they help investors understand why market risk can rise during periods of economic stress.


In [ ]:
# Chart 6: Correlation heatmap using Matplotlib.
corr_cols = [
    "fed_funds_rate",
    "inflation_yoy",
    "unemployment_rate",
    "real_interest_rate",
    "sp500_return_mom",
    "stock_minus_cash_gap",
    "high_inflation",
    "recession"
]

corr_matrix = econ[corr_cols].dropna().corr()

plt.figure(figsize=(9, 7))
plt.imshow(corr_matrix)
plt.colorbar(label="Correlation")
plt.xticks(range(len(corr_cols)), corr_cols, rotation=45, ha="right")
plt.yticks(range(len(corr_cols)), corr_cols)
plt.title("Correlation Heatmap of Macro and Market Variables")
plt.tight_layout()
plt.show()


### Chart 6 Interpretation

The correlation heatmap shows the relationships among macro and market variables. Macro variables such as the Fed Funds Rate, inflation, unemployment, and the real interest rate have clearer relationships with each other than with monthly S&P 500 returns. This is consistent with the later model result: monthly stock returns are noisy and hard to explain using only simple macro variables.

For the target audience, the key lesson is that macro indicators are useful for context, but they are not a simple stock-market prediction machine.


In [ ]:
# Chart 7: Investor comparison by real-rate environment.
chart7_data = real_rate_summary[[
    "avg_sp500_monthly_return",
    "avg_monthly_real_cash_return",
    "avg_stock_minus_cash_gap"
]].dropna()

x = np.arange(len(chart7_data.index))
width = 0.25

plt.figure(figsize=(12, 6))
plt.bar(x - width, chart7_data["avg_sp500_monthly_return"], width, label="Avg S&P 500 Monthly Return")
plt.bar(x, chart7_data["avg_monthly_real_cash_return"], width, label="Avg Monthly Real Cash Return Proxy")
plt.bar(x + width, chart7_data["avg_stock_minus_cash_gap"], width, label="Avg Stock-Minus-Cash Gap")
plt.title("Stock and Cash Returns by Real Interest Rate Environment")
plt.xlabel("Real Rate Environment")
plt.ylabel("Monthly Percent")
plt.xticks(x, chart7_data.index, rotation=15)
plt.legend()
plt.grid(True, axis="y")
plt.show()


### Chart 7 Interpretation

This chart compares stock returns, the real cash return proxy, and the stock-minus-cash gap across real-rate environments. The cash proxy is negative on average when real rates are negative and positive when real rates are positive. In the available S&P 500 sample, stocks have positive average monthly returns in both real-rate environments, so the stock-minus-cash gap remains positive.

For beginner investors, the practical message is that cash becomes more defensible when real rates are positive. When real rates are negative, holding cash may feel safe, but it can lose purchasing power.


In [ ]:
# Chart 8: Stock-minus-cash gap by inflation environment.
chart8_data = inflation_environment_summary[["avg_stock_minus_cash_gap"]].dropna()

plt.figure(figsize=(10, 6))
plt.bar(chart8_data.index, chart8_data["avg_stock_minus_cash_gap"])
plt.title("Average Stock-Minus-Cash Gap by Inflation Environment")
plt.xlabel("Inflation Environment")
plt.ylabel("Average Monthly Stock-Minus-Cash Gap (%)")
plt.grid(True, axis="y")
plt.show()


### Chart 8 Interpretation

This chart shows the average stock-minus-cash gap by inflation environment. In this sample, the gap is positive in both high-inflation and low/moderate-inflation months, but it is not a perfect trading signal. The important point is not that stocks always win, but that inflation changes the benchmark cash must beat.

For beginner investors, the chart emphasizes relative performance. The question is not only whether stocks go up, but whether stocks compensate investors for taking risk compared with holding cash.


## 9. Model 1: Linear Regression for S&P 500 Monthly Returns

The first model asks whether macro variables help explain monthly S&P 500 returns. The target variable is the monthly S&P 500 return. The features are the Federal Funds Rate, inflation, unemployment, the monthly rate change, recession status, and a few investor-oriented dummies. We deliberately leave out `real_interest_rate` here because it equals `fed_funds_rate - inflation_yoy`; including it alongside both of those columns would make the regression's coefficients unidentifiable.

This model is not expected to perfectly predict stock returns. Monthly stock returns are noisy. The goal is to test whether the model setup is economically reasonable and whether macro variables have some explanatory relationship with market returns.

In [ ]:
# We deliberately leave out real_interest_rate here. It is defined as
# fed_funds_rate - inflation_yoy, so including it alongside both of those columns
# would make the design matrix rank-deficient and the coefficients unidentifiable.
linear_features = [
    "fed_funds_rate",
    "inflation_yoy",
    "unemployment_rate",
    "rate_change",
    "negative_real_rate",
    "rate_hiking_month",
    "high_inflation",
    "recession"
]
linear_target = "sp500_return_mom"

linear_data = econ[linear_features + [linear_target]].dropna()
X = linear_data[linear_features]
y = linear_data[linear_target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

linear_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])

linear_model.fit(X_train, y_train)
y_pred = linear_model.predict(X_test)

linear_metrics = pd.DataFrame({
    "Metric": ["R-squared", "MAE", "RMSE"],
    "Value": [
        r2_score(y_test, y_pred),
        mean_absolute_error(y_test, y_pred),
        np.sqrt(mean_squared_error(y_test, y_pred))
    ]
}).round(4)

linear_metrics

### Model 1 Metrics Interpretation

The linear regression has a negative test R-squared, with an MAE of about 4.35 percentage points and an RMSE of about 6.39 percentage points. A negative test R-squared means the model performs worse than simply predicting the average return in the test set. This is not surprising because monthly S&P 500 returns are very noisy, and the available market sample is limited.

For the target audience, this is still a useful result. It warns beginner investors not to treat simple macro variables as a reliable short-term stock-return forecasting tool. Macro data can provide context, but it does not easily predict monthly equity returns.


In [ ]:
# Coefficient table for the linear regression model.
linear_coefficients = pd.DataFrame({
    "feature": linear_features,
    "coefficient": linear_model.named_steps["model"].coef_
}).sort_values("coefficient", ascending=False)

linear_coefficients.round(4)


### Model 1 Coefficient Interpretation

The coefficients describe the estimated relationship between each feature and monthly S&P 500 returns, holding the other variables constant. For example, the coefficient on `inflation_yoy` is negative, which suggests that higher inflation is associated with lower monthly stock returns in this fitted model. The coefficient on `fed_funds_rate` is positive in the model output, but this should not be interpreted as proof that rate hikes cause stock returns to rise.

Because the model has weak out-of-sample performance, the coefficients should be read as exploratory relationships rather than causal effects. For beginner investors, the main takeaway is humility: the stock market is hard to predict month to month, even when the macro story sounds reasonable.


## 10. Model 2: Logistic Regression for High Inflation Classification

The second model predicts whether a month is a **high-inflation month**, defined as year-over-year inflation above 3%. High inflation is directly relevant to beginner investors because it erodes the purchasing power of cash, which makes it a natural binary outcome for the project's question.

The target variable is `high_inflation`. A subtle but important modeling choice: because `real_interest_rate = fed_funds_rate - inflation_yoy`, any inflation-derived feature would let the model mechanically rebuild inflation and "predict" the target for a circular reason. We therefore exclude `real_interest_rate` and `negative_real_rate` and use only independent signals — the Federal Funds Rate, unemployment, the monthly rate change, stock-market returns, and policy dummies — so the metrics reflect a genuine classification problem.

In [ ]:
# IMPORTANT: the target high_inflation is a threshold on inflation_yoy, and
# real_interest_rate = fed_funds_rate - inflation_yoy. If we included real_interest_rate
# (or negative_real_rate, which is also derived from it), the model could mechanically
# rebuild inflation from its own features and score near-perfectly for a circular reason.
# We therefore exclude all inflation-derived features and predict high inflation only
# from independent policy, labor-market, and stock-market signals.
logit_features = [
    "fed_funds_rate",
    "unemployment_rate",
    "rate_change",
    "sp500_return_mom",
    "sp500_return_yoy",
    "rate_hiking_month",
    "recession"
]
logit_target = "high_inflation"

logit_data = econ[logit_features + [logit_target]].dropna()
X = logit_data[logit_features]
y = logit_data[logit_target].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

logit_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

logit_model.fit(X_train, y_train)
y_pred = logit_model.predict(X_test)
y_prob = logit_model.predict_proba(X_test)[:, 1]

logit_metrics = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score", "ROC AUC"],
    "Value": [
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred, zero_division=0),
        recall_score(y_test, y_pred, zero_division=0),
        f1_score(y_test, y_pred, zero_division=0),
        roc_auc_score(y_test, y_prob)
    ]
}).round(4)

logit_metrics

### Model 2 Metrics Interpretation

The logistic regression performs better than the stock-return regression. Accuracy is about 0.68, and ROC AUC is about 0.80. ROC AUC around 0.80 means the model has a reasonable ability to separate high-inflation months from low or moderate inflation months in the test data.

For the target audience, this model is more directly useful than the stock-return model because inflation directly affects cash purchasing power. The model does not perfectly forecast inflation, but it shows that macro and market features can help classify inflation regimes.


In [ ]:
# Confusion matrix for the high-inflation classification model.
conf_matrix = pd.DataFrame(
    confusion_matrix(y_test, y_pred, labels=[0, 1]),
    index=["Actual Low/Moderate Inflation", "Actual High Inflation"],
    columns=["Predicted Low/Moderate Inflation", "Predicted High Inflation"]
)
conf_matrix


### Confusion Matrix Interpretation

The confusion matrix shows that the model correctly classifies 9 low/moderate-inflation months and 6 high-inflation months in the test set. It misclassifies 4 low/moderate-inflation months as high inflation and 3 high-inflation months as low/moderate inflation. This is a reasonable but imperfect classification result.

For beginner investors, false negatives are important. If the model misses a high-inflation month, an investor may overestimate how attractive cash is. This connects directly to the project’s main point: inflation risk matters for cash holders.


In [ ]:
# Coefficient table for the logistic regression model.
logit_coefficients = pd.DataFrame({
    "feature": logit_features,
    "coefficient": logit_model.named_steps["model"].coef_[0]
}).sort_values("coefficient", ascending=False)

logit_coefficients.round(4)


### Model 2 Coefficient Interpretation

For logistic regression, positive coefficients increase the log-odds of a high-inflation month, while negative coefficients reduce the log-odds, holding other variables constant. The largest positive coefficient is `rate_change`, meaning that monthly increases in the Fed Funds Rate are strongly associated with high-inflation environments in this model. `fed_funds_rate` is also positive, which makes economic sense because policy rates are often higher when inflation pressure is high.

The coefficient on `unemployment_rate` is negative in this fitted model, suggesting that lower unemployment is associated with a higher probability of high inflation. This matches the idea that tight labor markets can coincide with stronger inflation pressure. For beginner investors, these coefficients help explain which macro signals may warn that cash returns need to be evaluated more carefully against inflation.


## 11. Export Cleaned Data and SQLite Tables

The final project also requires a separate SQL file or SQL notebook. To make the SQL work easy to run, this section exports the cleaned final dataset and creates a SQLite database with separate tables for the original datasets and the final merged dataset.


In [ ]:
# Save cleaned data as a CSV file.
econ.to_csv("final_project_cleaned_data.csv", index=False)

# Create a SQLite database for the separate SQL queries.
conn = sqlite3.connect("final_project_macro.db")

fed_funds_m.to_sql("fed_funds", conn, if_exists="replace", index=False)
unemployment_m.to_sql("unemployment", conn, if_exists="replace", index=False)
cpi_m.to_sql("cpi", conn, if_exists="replace", index=False)
sp500_m.to_sql("sp500", conn, if_exists="replace", index=False)
recession_m.to_sql("recession", conn, if_exists="replace", index=False)
econ.to_sql("econ_final", conn, if_exists="replace", index=False)

conn.close()

print("Exported final_project_cleaned_data.csv")
print("Created final_project_macro.db")


## 12. Conclusion

This project shows that nominal interest rates alone do not fully describe the investor experience. Cash is more attractive when interest rates are high relative to inflation, not simply when nominal rates are high. The real interest rate gives a better view of cash purchasing power.

The most useful insight for beginner investors is that **real interest rates matter more than nominal interest rates**. When inflation exceeds the Federal Funds Rate, cash may lose purchasing power even when savings yields appear attractive. The stock-minus-cash gap also shows that the relative attractiveness of stocks and cash changes across inflation and real-rate environments.

The S&P 500 analysis adds a market perspective. The linear regression model shows that monthly stock returns are difficult to explain using only macro variables, which is realistic because equity returns are noisy in the short run. The high-inflation logistic regression is more directly tied to the investor question because inflation affects the real value of cash returns.

Overall, the main takeaway is that macro indicators are useful for context, but they are not perfect trading signals. A high interest rate environment can still be unattractive for cash if inflation is also high. Similarly, stock returns may react to macro conditions, but month-to-month prediction remains difficult.

If there were more time, useful additions would include bond ETF returns, Treasury bill yields, consumer sentiment, sector-level stock returns, and money market fund yields. These additions would allow a more direct comparison of cash, bonds, and stocks across different macroeconomic environments.
